## Convert Archive Content in SQLLite
Take each row in the existing PBSB database and convert it to a flat file. Discard unwanted fields. Change some parts of the image tags. We will use these files to dynamically generate website content using 11ty.


### Imports

In [39]:
import sqlite3
import csv
import pandas as pd
import os
import json
import re

### Read Product Data from Database
First, read the products from the PBSB database and write into a flat CSV output file.

In [40]:
# Connect to the SQLite database
conn = sqlite3.connect('C:\\Users\\simon\\Projects\\PBSB\\PBSB-Main.db')
cursor = conn.cursor()

# Query the data
# this is much simpler than the blog post extraction because everything we need is already in the table and
# each item is only associated with one category

cursor.execute("""SELECT 
    p.*
    FROM Product p;""")

fileName = 'archives.csv'

# Write to CSV
with open(fileName, 'w', newline='') as csv_file:
    writer = csv.writer(csv_file, quoting=csv.QUOTE_ALL, lineterminator='\n')
    # For this conversion, we skip the headers (can be useful during development)
    # writer.writerow([i[0] for i in cursor.description])
    # Write data
    writer.writerows(cursor.fetchall())

conn.close()

### Process the CSV File
Now switch to Pandas for easier processing of the data. Convert Id and CategoryId fields to integer.

In [41]:
# Read the CSV file making sure we omit the header row
df = pd.read_csv(fileName, encoding='windows-1252', header=None, dtype={0: "Int64", 6: "Int64", 8: "Int64"})

In [42]:
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,13,Susannah,Susannah. Pencil on paper 11x14in. 2008,Pencil portrait of Susannah by Simon Bland,Susannah.jpg,0.0,3,People,1,1,NaN,NaN,NaN
1,14,Catherine,Catherine. Pencil on paper 11x14in. 2006,Pencil portrait of Catherine by Simon Bland,Swing.jpg,0.0,3,People,1,1,NaN,NaN,NaN
2,16,Kate,Kate. Oil on linen 16x16in. 2007 (from life),Oil portrait of Kate painted from life by Simo...,Kate2.jpg,0.0,3,People,1,1,16x16,Oil on linen.,NaN
3,17,Katherine and Gus,Katherine and Gus. Oil on linen 24x30in. 2010,Oil portrait of Katherine and Gus by Simon Bland,Shaw.jpg,0.0,3,People,1,1,NaN,NaN,NaN
4,18,Gemma and family,Gemma and family. Oil on linen 30x40in. 2009,Oil portrait of Gemma by Simon Bland,Gemma.jpg,0.0,3,People,1,1,NaN,NaN,NaN


Delete unnecessary fields. Note: the Id field helps provide chronological order.

In [43]:
df = df.drop(df.columns[[5, 6, 8, 9, 12]], axis=1)

df.head()

,0,1,2,3,4,7,10,11
0,13,Susannah,Susannah. Pencil on paper 11x14in. 2008,Pencil portrait of Susannah by Simon Bland,Susannah.jpg,People,NaN,NaN
1,14,Catherine,Catherine. Pencil on paper 11x14in. 2006,Pencil portrait of Catherine by Simon Bland,Swing.jpg,People,NaN,NaN
2,16,Kate,Kate. Oil on linen 16x16in. 2007 (from life),Oil portrait of Kate painted from life by Simo...,Kate2.jpg,People,16x16,Oil on linen.
3,17,Katherine and Gus,Katherine and Gus. Oil on linen 24x30in. 2010,Oil portrait of Katherine and Gus by Simon Bland,Shaw.jpg,People,NaN,NaN
4,18,Gemma and family,Gemma and family. Oil on linen 30x40in. 2009,Oil portrait of Gemma by Simon Bland,Gemma.jpg,People,NaN,NaN


### Wrangle the Data

URLs can be reconstructed easily when compiling the .liquid file, so we do not need to do it here. However, some fields can contain NaN which is better replaced by blank space,.

In [44]:
df = df.fillna("")

### Add Column Names


In [45]:
df.columns = ['Id', 'Title', 'Description', 'Alt', 'Image', 'Category', 'Medium', 'Ground']

### Remove Unwanted Records
Not all the records are needed and some paintings haven't aged well. Id 254 may be a duplicate or corrupted. Best to strip them out and have a more streamlined website content.

In [46]:
ids_to_delete = [
    22, 152, 151, 247, 206, 118, 141, 290, 289, 280,
    283, 274, 242, 264, 237, 252, 251, 248, 249, 245,
    241, 240, 238, 236, 233, 232, 227, 223, 219, 190,
    212, 214, 196, 150, 137, 120, 113, 71, 29, 285, 254
]

df = df[~df['Id'].isin(ids_to_delete)]


### Correct Duplicate Titles
Duplicates cause problems because Title is used to name the .liquid file

In [47]:
#df.loc[df['Id'] == 285, 'Title'] = "Abstract Rooster"

### Add Slug

This is used to construct the permalink

In [48]:
df['Slug'] = (
    df.iloc[:, 1]
      .astype(str)
      .str.lower()
      .str.strip()
      .str.replace('[!()]', '', regex=True)
      .str.replace(r'\s+', '-', regex=True)
      .str.replace(r'-+', '-', regex=True)
      .str.strip('-')
)

C:\Users\simon\AppData\Local\Temp\ipykernel_18152\3551493937.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Slug'] = (


### Add Previous and Next Links

Group by category and sort by Id

In [49]:
# Sort by Category + Id
df = df.sort_values(["Category", "Id"], ascending=[True, False]).reset_index(drop=True)

# Compute previous/next within each category
df["PrevSlug"] = df.groupby("Category")["Slug"].shift(1)
df["PrevId"]   = df.groupby("Category")["Id"].shift(1)

df["NextSlug"] = df.groupby("Category")["Slug"].shift(-1)
df["NextId"]   = df.groupby("Category")["Id"].shift(-1)


Convert the ids into permalink paths

In [50]:
def make_link(row, which):
    slug = row[f"{which}Slug"]
    id_  = row[f"{which}Id"]
    if pd.isna(slug):
        return None
    cat = row["Category"].lower()
    return f"/archives/{cat}/{int(id_)}/{slug}/"

df["PrevLink"] = df.apply(lambda r: make_link(r, "Prev"), axis=1)
df["NextLink"] = df.apply(lambda r: make_link(r, "Next"), axis=1)


### Final Visual Inspection of the Data
Last chance to visually inspect the data before handing it over to 11ty.

In [51]:
df.head()

,Id,Title,Description,Alt,Image,Category,Medium,Ground,Slug,PrevSlug,PrevId,NextSlug,NextId,PrevLink,NextLink
0,300,Maddie M,An oil portrait of a King Charles spaniel. Don...,Oil portrait of a King Charles spaniel,MaddieM.jpg,Dogs,14x11,Oil on linen panel,maddie-m,NaN,<NA>,izzy,298,None,/archives/dogs/298/izzy/
1,298,Izzy,"A portrait of Izzy, an English springer spaniel.",An oil painting of an English springer spaniel...,Izzy.jpg,Dogs,12x10,Oil on linen panel,izzy,maddie-m,300,sweetie,294,/archives/dogs/300/maddie-m/,/archives/dogs/294/sweetie/
2,294,Sweetie,An abstract painting of our dog Sweetie in yel...,An abstract oil painting of a dog with yellow ...,Sweetie.jpg,Dogs,14x14,Oil on linen,sweetie,izzy,298,tate,288,/archives/dogs/298/izzy/,/archives/dogs/288/tate/
3,288,Tate,A portrait of a French Bulldog. This was paint...,A portrait of a French Bulldog,Tate14x11.jpg,Dogs,14x11,Oil on linen,tate,sweetie,294,hershey-and-cinnamon,277,/archives/dogs/294/sweetie/,/archives/dogs/277/hershey-and-cinnamon/
4,277,Hershey and Cinnamon,A pastel painting of two shelties done from th...,A pastel painting of two shelties done from th...,Hershey-and-Cinnamon.jpg,Dogs,22x18,Pastel on MiTeintes Touch,hershey-and-cinnamon,tate,288,louis,273,/archives/dogs/288/tate/,/archives/dogs/273/louis/


### Write Out the Data to a Unix-type CSV file (has LF line endings, not CRLF)

Do this so we have an easily readable copy of the output on hand for reference. It is otherwise not used for the website.

In [52]:
# Write the cleaned-up data to the CSV file. Make sure there is a header row!

df.to_csv(fileName, index=False, header=True, lineterminator='\n')

## Write Out Individual Archives Files

Create the individual files that will contain each archive:
- We group by category so that we essentially have a set of paginated lists
- URLs that existed in the old website are not strictly preserved for now - I have removed the detail pages so that I can avoid having to reconstruct the navigation
- Category information is preserved in the header, so we can segregate the archives by type like we did with blog post categories
- Slugs have been constructed
  

In [53]:
# Output directory
output_dir = "archives"
os.makedirs(output_dir, exist_ok=True)

Data has already been cleaned and formatted so it can be written directly to an output file. Note that each file will only
have header content.

In [54]:
# Write each row to its own text file
for idx, row in df.iterrows():

    file_path = os.path.join(output_dir, str(row['Slug']) + ".liquid")

    try:
        with open(file_path, "w", encoding="utf-8") as f:
            f.write("--- \n")
            f.write("layout: archive.njk" + "\n")
            f.write("title: \"" + str(row['Title']).replace('"', '\\"') + "\"\n")
            f.write(f"id: {row['Id']}\n")
            f.write("permalink: /archives/" + str(row['Category']).lower() + "/"  +str(row['Id']) + "/" + str(row['Slug']) + "/ \n")
            f.write(f"prev: {row['PrevLink'] or 'null'}\n")
            f.write(f"next: {row['NextLink'] or 'null'}\n")
            f.write("image: /images/" + str(row['Category']).lower() + "/" + str(row['Image']) + "\n")
            f.write("thumb: /images/" + str(row['Category']).lower() + "/thumbs/" + str(row['Image']) + "\n")
            f.write("alt:  \"" + str(row['Alt']) + "\"\n")
            f.write("medium:  \"" + str(row['Medium']) + "\"\n")
            f.write("ground:  \"" + str(row['Ground']) + "\"\n")
            f.write("description: \"" + str(row['Description']).replace('"', '\\"') + "\"\n")
            #f.write("category: \"" + str(row['Category']) + "\"\n")

            # archive categories as YAML list
            f.write("archiveCategories:\n")
            f.write(f" - {row['Category']}\n")
            
            f.write("tags: archive" + "\n")
            f.write("--- \n")
            f.write("\n")
    except OSError as e:
        print(f"Error writing file {file_path}: {e}")

print(f"{len(df)} files written to '{output_dir}'")

152 files written to 'archives'
